# Notebook 03: Clinical Text Preprocessing

**Author:** Anthony Amit Biswas

## What this notebook does

Cleans de-identification placeholders and formatting noise while preserving clinically meaningful terminology, punctuation, and measurements.


**Importing Libraries**

In [ ]:
# Import Required Libraries

# These libraries are used throughout the preprocessing pipeline.

import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 500)

print("Libraries imported successfully.")

**Loading the Dataset**

In [ ]:
# Load the ICU Discharge Summary Cohort

# The cleaned ICU cohort created in Notebook 01 serves as the
# starting point for all subsequent NLP preprocessing.

DATA_PATH = "/content/drive/MyDrive/Dissertation/outputs/"

icu_notes = pd.read_csv(
    DATA_PATH + "icu_discharge_summary_cohort.csv.gz",
    compression="gzip",
    low_memory=False
)

print("-" * 70)
print("ICU COHORT LOADED")
print("-" * 70)

print(f"Rows    : {icu_notes.shape[0]:,}")
print(f"Columns : {icu_notes.shape[1]}")

**Keeping Only NLP Columns**

In [ ]:
# Creating the NLP Dataset

# Only the identifiers and discharge-summary text are required
# for the NLP preprocessing pipeline.

nlp_notes = icu_notes[
    [
        "note_id",
        "subject_id",
        "hadm_id",
        "text"
    ]
].copy()

display(nlp_notes.head())

**Examine Raw Text**

In [ ]:
# Display a Raw Discharge Summary

# Before preprocessing, one discharge summary is displayed to
# understand the formatting and identify elements that require
# cleaning.

sample_text = nlp_notes.loc[0, "text"]

print(sample_text[:5000])

**Counting De-identification Placeholders**

In [ ]:
# Counteing De-identification Placeholders

# MIMIC-IV replaces identifiable information with placeholders
# such as "___".

# These placeholders do not carry clinical meaning and will be
# removed during preprocessing.


placeholder_pattern = r"___"

placeholder_counts = (
    nlp_notes["text"]
    .str.count(placeholder_pattern)
)

print("-" * 70)
print("DE-IDENTIFICATION PLACEHOLDERS")
print("-" * 70)

print(f"Total placeholders : {placeholder_counts.sum():,}")

print(f"Average per note   : {placeholder_counts.mean():.2f}")

print(f"Maximum in one note: {placeholder_counts.max()}")

**Creating a Preprocessing Function**

In [ ]:
# Clinical Text Preprocessing Function

# This function performs conservative preprocessing suitable for
# ICU discharge summaries.

# The aim is to improve text consistency without removing clinically
# meaningful information.

# Preprocessing steps:
# 1. Remove MIMIC de-identification placeholders.
# 2. Reduce repeated horizontal spaces.
# 3. Remove spaces at the beginning and end of each line.
# 4. Reduce excessive blank lines.
# 5. Preserve clinical terminology, punctuation, numbers, abbreviations
#    and section boundaries.

# De-identification placeholders are removed rather than replaced with
# an artificial token because they do not contain clinically relevant
# information for the complication-extraction task.


def preprocess_clinical_text(text):

    # Returning an empty string if the note is missing.
    if pd.isna(text):
        return ""

    # Converting the value to a string to ensure that regular-expression
    # operations can be applied safely.
    text = str(text)

    # Removing MIMIC de-identification placeholders.
    #
    # The pattern matches any continuous sequence containing three or
    # more underscore characters. A single space is inserted so that
    # surrounding words do not become joined together.
    text = re.sub(r"_{3,}", " ", text)

    # Replacing repeated horizontal whitespace with one space.
    #
    # This removes unnecessary spacing while preserving newline
    # characters, which are important for section and paragraph structure.
    text = re.sub(r"[ \t]{2,}", " ", text)

    # Removing unnecessary spaces from the beginning and end of each line.

    # Empty lines are retained at this stage because they help preserve
    # boundaries between clinical sections.
    cleaned_lines = [
        line.strip()
        for line in text.splitlines()
    ]

    text = "\n".join(cleaned_lines)

    # Replacing three or more consecutive line breaks with two line breaks.
    #
    # This removes excessive blank space while preserving paragraph and
    # section separation.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Removing whitespace from the beginning and end of the complete note.
    text = text.strip()

    return text

**Applying the Function**

In [ ]:
# Apply Clinical Text Preprocessing

nlp_notes["clean_text"] = (
    nlp_notes["text"]
    .apply(preprocess_clinical_text)
)

print("Preprocessing completed successfully.")

**Comparing Before and After**

In [ ]:
# Compare Raw and Preprocessed Text

sample_index = 0

print("-" * 80)
print("RAW TEXT")
print("-" * 80)

print(nlp_notes.loc[sample_index, "text"][:2000])

print("\n")
print("-" * 80)
print("PREPROCESSED TEXT")
print("-" * 80)

print(nlp_notes.loc[sample_index, "clean_text"][:2000])

**Verify Cleaning**

In [ ]:
# Validating the Preprocessing

remaining_placeholders = (
    nlp_notes["clean_text"]
    .str.count(r"_{3,}")
    .sum()
)

remaining_redacted_tokens = (
    nlp_notes["clean_text"]
    .str.count(r"\[REDACTED\]")
    .sum()
)

missing_clean_text = (
    nlp_notes["clean_text"]
    .isna()
    .sum()
)

empty_clean_text = (
    nlp_notes["clean_text"]
    .str.strip()
    .eq("")
    .sum()
)

print("-" * 70)
print("POST-PREPROCESSING VALIDATION")
print("-" * 70)

print(f"Original placeholders remaining : {remaining_placeholders:,}")
print(f"[REDACTED] tokens remaining    : {remaining_redacted_tokens:,}")
print(f"Missing cleaned notes           : {missing_clean_text:,}")
print(f"Empty cleaned notes             : {empty_clean_text:,}")

## **Preprocessing Validation**

The final preprocessing pipeline successfully removed all MIMIC-IV de-identification placeholders without introducing artificial replacement tokens.

No original underscore placeholders or `[REDACTED]` tokens remained after preprocessing. In addition, no discharge summaries became missing or empty.

The preprocessing strategy was deliberately conservative. Clinical section headings, punctuation, abbreviations, medication dosages, laboratory measurements and numerical values were preserved because these elements may provide important contextual information for downstream clinical NLP.

Some sentence fragments remain grammatically incomplete where identifying information was removed. This is an expected feature of the de-identified MIMIC-IV notes and was considered preferable to introducing artificial tokens that might add noise to the NLP models.

## **3. Quantitative Assessment of Preprocessing**

To ensure that preprocessing did not remove excessive clinical content, the raw and cleaned discharge summaries were compared using character and word counts.

The analysis measures:

- Raw and cleaned character counts
- Raw and cleaned word counts
- Absolute reduction in text length
- Percentage reduction in text length

This provides a quantitative validation of the conservative preprocessing strategy.

**Calculating Raw and Cleaned Text Lengths**

In [ ]:
# Calculating Raw and Cleaned Text Lengths

nlp_notes["raw_character_count"] = (
    nlp_notes["text"]
    .fillna("")
    .str.len()
)

nlp_notes["clean_character_count"] = (
    nlp_notes["clean_text"]
    .fillna("")
    .str.len()
)

nlp_notes["raw_word_count"] = (
    nlp_notes["text"]
    .fillna("")
    .str.split()
    .str.len()
)

nlp_notes["clean_word_count"] = (
    nlp_notes["clean_text"]
    .fillna("")
    .str.split()
    .str.len()
)

nlp_notes["character_reduction"] = (
    nlp_notes["raw_character_count"]
    - nlp_notes["clean_character_count"]
)

nlp_notes["word_reduction"] = (
    nlp_notes["raw_word_count"]
    - nlp_notes["clean_word_count"]
)

nlp_notes["character_reduction_percentage"] = (
    nlp_notes["character_reduction"]
    / nlp_notes["raw_character_count"]
    * 100
)

nlp_notes["word_reduction_percentage"] = (
    nlp_notes["word_reduction"]
    / nlp_notes["raw_word_count"]
    * 100
)

print("-" * 75)
print("EFFECT OF FINAL CLINICAL TEXT PREPROCESSING")
print("-" * 75)

print(
    f"Median raw characters       : "
    f"{nlp_notes['raw_character_count'].median():,.0f}"
)

print(
    f"Median cleaned characters   : "
    f"{nlp_notes['clean_character_count'].median():,.0f}"
)

print(
    f"Median character reduction  : "
    f"{nlp_notes['character_reduction_percentage'].median():.2f}%"
)

print()

print(
    f"Median raw words            : "
    f"{nlp_notes['raw_word_count'].median():,.0f}"
)

print(
    f"Median cleaned words        : "
    f"{nlp_notes['clean_word_count'].median():,.0f}"
)

print(
    f"Median word reduction       : "
    f"{nlp_notes['word_reduction_percentage'].median():.2f}%"
)

## **Effect of Preprocessing on Text Length**

The final preprocessing pipeline produced a modest reduction in note length.

The median discharge summary decreased from 11,604 to 11,005 characters, corresponding to a median character reduction of 5.00%. The median word count decreased from 1,755 to 1,703 words, corresponding to a median word reduction of 2.86%.

These results indicate that preprocessing removed de-identification placeholders and unnecessary whitespace while retaining most of the original clinical content. The relatively small reduction in word count supports the use of a conservative preprocessing strategy that preserves clinical terminology, numerical measurements, medication information, punctuation and document structure.

**Preparing the Final Preprocessed Dataset**

In [ ]:
# Preparing the Final Preprocessed Dataset

# Only the identifiers, original note text and cleaned note text are retained.
# Temporary variables used for preprocessing validation are excluded from the
# final output dataset.

preprocessed_notes = nlp_notes[
    [
        "note_id",
        "subject_id",
        "hadm_id",
        "text",
        "clean_text"
    ]
].copy()

print("-" * 70)
print("FINAL PREPROCESSED DATASET")
print("-" * 70)

print(f"Rows    : {preprocessed_notes.shape[0]:,}")
print(f"Columns : {preprocessed_notes.shape[1]:,}")

display(preprocessed_notes.head())

**Save the Final Preprocessed Dataset**

In [ ]:
# Save the Final Preprocessed Dataset

preprocessed_output_path = (
    "/content/drive/MyDrive/Dissertation/outputs/"
    "icu_discharge_summaries_preprocessed.csv.gz"
)

preprocessed_notes.to_csv(
    preprocessed_output_path,
    index=False,
    compression="gzip"
)

print("-" * 70)
print("DATASET SAVED SUCCESSFULLY")
print("-" * 70)

print(f"Output path: {preprocessed_output_path}")

**Verifying the saved file**

In [ ]:
# Verify the Saved Dataset

saved_preprocessed_notes = pd.read_csv(
    preprocessed_output_path,
    compression="gzip"
)

print("-" * 70)
print("SAVED DATASET VERIFICATION")
print("-" * 70)

print(f"Rows loaded              : {saved_preprocessed_notes.shape[0]:,}")
print(f"Columns loaded           : {saved_preprocessed_notes.shape[1]:,}")
print(f"Missing cleaned notes    : {saved_preprocessed_notes['clean_text'].isna().sum():,}")
print(
    f"Empty cleaned notes      : "
    f"{saved_preprocessed_notes['clean_text'].fillna('').str.strip().eq('').sum():,}"
)
print(
    f"Duplicate note IDs       : "
    f"{saved_preprocessed_notes['note_id'].duplicated().sum():,}"
)